# DSA_02 — Build Precomputed Public-Demo Artifacts

**Purpose.** Execute the already validated local pipeline for the small set of
real demo origins selected in DSA_01.

**Important:** This is the computationally heavy notebook. It runs locally,
where the frozen models exist. The resulting Parquet/JSON files are the only
artifacts required by the public Streamlit deployment.

In [1]:
# Import libraries
from pathlib import Path
import sys, json, yaml, pandas as pd

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / "configs" / "decision_support_app.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/decision_support_app.yaml')

In [3]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))

In [4]:
from src.ontario_peak_risk.decision_support.common import resolve
from src.ontario_peak_risk.decision_support.demo_builder import (
    build_prediction_demo,
    build_weather_scenario_demo,
)
from src.ontario_peak_risk.decision_support.contracts import (
    validate_predictions,
    validate_scenarios,
)

In [5]:
demo_origins_path = resolve(PROJECT_ROOT, cfg["paths"]["demo_origins"])
if not demo_origins_path.exists():
    raise FileNotFoundError("Run DSA_01 first: demo_origins.csv is missing.")

demo_origins = pd.read_csv(demo_origins_path, parse_dates=["forecast_origin"])
display(demo_origins)

,forecast_origin,demand_history_status,weather_grid_status,fsa_count
0,2026-01-08 00:00:00,PASS,PASS,6
1,2026-03-05 12:00:00,PASS,PASS,6
2,2026-04-30 23:00:00,PASS,PASS,6


In [6]:
predictions = build_prediction_demo(cfg, PROJECT_ROOT, demo_origins)

validation = validate_predictions(
    predictions,
    cfg["application"]["fsas"],
    int(cfg["application"]["horizons"]),
    float(cfg["application"]["official_peak_threshold"]),
)

if validation["status"].eq("FAIL").any():
    display(validation[validation["status"].eq("FAIL")])
    raise ValueError("Prediction demo validation failed.")

pred_path = resolve(PROJECT_ROOT, cfg["paths"]["demo_predictions"])
pred_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_parquet(pred_path, index=False)

print("Prediction rows:", len(predictions))
print("Prediction validation: PASS")

[1/3] Generating predictions for 2026-01-08 00:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


[2/3] Generating predictions for 2026-03-05 12:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


[3/3] Generating predictions for 2026-04-30 23:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


Prediction rows: 432
Prediction validation: PASS


In [7]:
scenarios = pd.DataFrame()

if cfg["demo"]["generate_weather_scenarios"]:
    scenarios = build_weather_scenario_demo(
        cfg,
        PROJECT_ROOT,
        demo_origins,
    )

    scenario_validation = validate_scenarios(
        scenarios,
        cfg["application"]["fsas"],
        int(cfg["application"]["horizons"]),
    )

    if scenario_validation["status"].eq("FAIL").any():
        display(scenario_validation[scenario_validation["status"].eq("FAIL")])
        raise ValueError("Scenario demo validation failed.")

    scenario_path = resolve(
        PROJECT_ROOT,
        cfg["paths"]["demo_weather_scenarios"],
    )
    scenarios.to_parquet(scenario_path, index=False)

    print("Scenario rows:", len(scenarios))
    print("Scenario validation: PASS")

Generating weather scenarios for 2026-01-08 00:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


Generating weather scenarios for 2026-03-05 12:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


Generating weather scenarios for 2026-04-30 23:00:00 ...


e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\src\ontario_peak_risk\inference_feature_builder\io.py:198: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  combined[col] = combined[update].combine_first(combined.get(col))


Scenario rows: 2160
Scenario validation: PASS


In [8]:
metadata = {
    "mode": "public_demo_precomputed",
    "forecast_origins": [
        str(x) for x in pd.to_datetime(demo_origins["forecast_origin"])
    ],
    "fsas": cfg["application"]["fsas"],
    "horizons": int(cfg["application"]["horizons"]),
    "official_peak_threshold": float(
        cfg["application"]["official_peak_threshold"]
    ),
    "models_loaded_by_public_app": False,
    "weather_scenarios_generated": not scenarios.empty,
}

metadata_path = resolve(PROJECT_ROOT, cfg["paths"]["demo_metadata"])
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("DSA_02 RESULT: COMPLETE")

DSA_02 RESULT: COMPLETE
